In [1]:
# Fix random seed
import numpy as np
import torch 
import random
def set_seed(seed):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x10b5d61f0>>
Traceback (most recent call last):
  File "/Users/lixiansheng/Desktop/Lhy_Machine_Learning-main/2022 ML/04 Sequence as input/.venv/lib/python3.9/site-packages/ipykernel/ipkernel.py", line 781, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 


In [ ]:
import os
import json
import torch
from pathlib import Path
from torch.utils.data import Dataset

class myDataset(Dataset):
    def __init__(self, data_dir, segment_len = 128):
        super().__init__()
        self.data_dir = data_dir
        self.segment_len = segment_len

        mapping_path = Path(data_dir) / 'mapping.json'
        mapping = json.load(mapping_path.open())
        # print(type(mapping))
        self.speaker2id = mapping['speaker2id']
        # print(self.speaker2id)

        metadata_path = Path(data_dir) / 'metadata.json'
        metadata = json.load(metadata_path.open())['speakers']
        # print(metadata)

        self.speaker_num = len(metadata.keys())
        self.data = []
        for speaker in metadata:
            for utterances in metadata[speaker]:
                self.data.append([utterances['feature_path'], self.speaker2id[speaker]])### 这里是接入列表
        # print(self.data)
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        feat_path, speaker = self.data[idx]
        # mel = torch.load(os.path.join(self.data_dir,feat_path))
        mel = torch.load(Path(self.data_dir) / feat_path)

        if len(mel) > self.segment_len:
            st = random.randint(0, len(mel) - self.segment_len)
            mel = mel[st: st+self.segment_len]
            mel = torch.tensor(mel, dtype=torch.float)
        else:
            mel = torch.tensor(mel, dtype=torch.float)
        speaker = torch.LongTensor([speaker])
        
        return mel, speaker

    def get_speaker_number(self):
        return self.speaker_num



        

In [ ]:
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, random_split

def collate_batch(batch):
    mel, speaker = zip(*batch)### list of tuples ---- > two tuples 1: tuple of (batch_size, context_len, 40) 2: tuple of (tensor.long)
    mel = pad_sequence(mel, batch_first=True, padding_value=-20)
    return mel, torch.FloatTensor(speaker).long()

def get_dataloader(data_dir, batch_size, n_workers):
    dataset = myDataset(data_dir)
    speaker_num = dataset.get_speaker_number()
    trainlen = int(len(dataset) * 0.9)
    lengths = [trainlen, len(dataset) - trainlen]
    trainset, validset = random_split(dataset=dataset, lengths=lengths)

    train_loader = DataLoader(trainset, batch_size=batch_size, shuffle=True, drop_last=True, num_workers=n_workers, pin_memory=True, collate_fn=collate_batch)

    valid_loader = DataLoader(validset, batch_size=batch_size, drop_last=True, num_workers=n_workers, pin_memory=True, collate_fn=collate_batch)

    return train_loader, valid_loader, speaker_num
    

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class Classifier(nn.Module):
    def __init__(self, d_model = 80, n_spks= 600):
        super().__init__()
        self.prenet = nn.Linear(40, d_model)
        self.encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, dim_feedforward=256, nhead=2)
        self.pre_layer = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Linear(d_model, n_spks)
        )
    def forward(self, x):
        ### x shape :(batch_size, context_len, 40)
        out = self.prenet(x)
        out = self.encoder_layer(out.permute(1,0,2))### input:(context_len, batch_size, 80); output: (context_len, batch_size, 80)
        out = out.transpose(0, 1)
        stats = out.mean(dim = 1) ### output: (batch_size,80)
        out = self.pre_layer(stats)### output: (batch_size, 600) logits
        return out

In [ ]:
import math
import torch.optim as Optimizer
from torch.optim.lr_scheduler import LambdaLR

def get_cosine_schedule_with_warmup(
        optimizer: Optimizer,
        num_warmup_steps: int,
        num_training_steps: int,
        num_cycles: float = 0.5,
        last_epoch: int = -1
):
    def lr_lambda(current_step):
        # warm up
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        # decadence
        progress = float(current_step - num_warmup_steps) / float(
            max(1, num_training_steps - num_warmup_steps)
        )
        return max(
            0.0, 0.5 * (1.0 + math.cos(math.pi * float(num_cycles) * 2.0 * progress))
        )
    return LambdaLR(optimizer, lr_lambda, last_epoch)

In [ ]:
def model_fn(batch, model, criterion, device):
    mels, labels = batch
    mels, labels = mels.to(device), labels.to(device)
    outs = model(mels)

    loss = criterion(outs, labels)### (batch_size, n_spks) and (batch_size, )

    preds = torch.argmax(outs, dim = 1)
    accuracy = (preds.detach() == labels).float().mean()

    return loss, accuracy

In [ ]:
from tqdm import tqdm

def valid(dataloader, model, criterion, device):
    model.eval()
    running_loss = 0.0
    running_accuracy = 0.0
    pbar = tqdm(total=len(dataloader.dataset), ncols=0, desc='Valid', unit = 'uttr')

    for i, batch in enumerate(dataloader):
        with torch.no_grad():
            loss, accuracy = model_fn(batch, model, criterion, device)
            running_loss += loss.item()
            running_accuracy += accuracy.item()

        pbar.update(dataloader.batch_size)
        pbar.set_postfix(loss=f"{running_loss / (i+1):.2f}",
			accuracy=f"{running_accuracy / (i+1):.2f}",)
        
    pbar.close()
    model.train()

    return running_accuracy / len(dataloader)

In [ ]:
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, random_split


def parse_args():
	"""arguments"""
	config = {
		"data_dir": "./Dataset_small",
		"save_path": "model.ckpt",
		"batch_size": 32,
		"n_workers": 0,
		"valid_steps": 2000,
		"warmup_steps": 1000,
		"save_steps": 10000,
		"total_steps": 70000,
	}

	return config


def main(
	data_dir,
	save_path,
	batch_size,
	n_workers,
	valid_steps,
	warmup_steps,
	total_steps,
	save_steps,
):
	"""Main function."""
	device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
	print(f"[Info]: Use {device} now!")

	train_loader, valid_loader, speaker_num = get_dataloader(data_dir, batch_size, n_workers)
	train_iterator = iter(train_loader)
	print(f"[Info]: Finish loading data!",flush = True)

	model = Classifier(n_spks=speaker_num).to(device)
	criterion = nn.CrossEntropyLoss()
	optimizer = AdamW(model.parameters(), lr=1e-3)
	scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
	print(f"[Info]: Finish creating model!",flush = True)

	best_accuracy = -1.0
	best_state_dict = None

	pbar = tqdm(total=valid_steps, ncols=0, desc="Train", unit=" step")

	for step in range(total_steps):
		# Get data
		try:
			batch = next(train_iterator)
		except StopIteration:
			train_iterator = iter(train_loader)
			batch = next(train_iterator)

		loss, accuracy = model_fn(batch, model, criterion, device)
		batch_loss = loss.item()
		batch_accuracy = accuracy.item()

		# Updata model
		loss.backward()
		optimizer.step()
		scheduler.step()###
		optimizer.zero_grad()

		# Log
		pbar.update()
		pbar.set_postfix(
			loss=f"{batch_loss:.2f}",
			accuracy=f"{batch_accuracy:.2f}",
			step=step + 1,
		)

		# Do validation
		if (step + 1) % valid_steps == 0:
			pbar.close()

			valid_accuracy = valid(valid_loader, model, criterion, device)

			# keep the best model
			if valid_accuracy > best_accuracy:
				best_accuracy = valid_accuracy
				best_state_dict = model.state_dict()

			pbar = tqdm(total=valid_steps, ncols=0, desc="Train", unit=" step")

		# Save the best model so far.
		if (step + 1) % save_steps == 0 and best_state_dict is not None:
			torch.save(best_state_dict, save_path)
			pbar.write(f"Step {step + 1}, best model saved. (accuracy={best_accuracy:.4f})")

	pbar.close()


if __name__ == "__main__":
	main(**parse_args())

In [ ]:
import json
import os
from pathlib import Path
from tqdm import tqdm
from torch.utils.data import Dataset

class InferenceDataset(Dataset):
    def __init__(self, data_dir):
        super().__init__()
        testdata_path = Path(data_dir) / 'testdata.json'
        metadata = json.load(testdata_path.open())
        self.data_dir = data_dir
        self.data = metadata['utterances']

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        utterance = self.data[idx]
        feat_path = utterance['feature_path']
        mel = torch.load(os.path.join(self.data_dir, feat_path))
        return feat_path, mel
    
def inference_collate_fn(batch):
    feat_paths, mels = zip(*batch)
    return feat_paths, torch.stack(mels)


In [ ]:
import json
import csv
from pathlib import Path
from tqdm.notebook import tqdm

import torch
from torch.utils.data import DataLoader

def parse_args():
	"""arguments"""
	config = {
		"data_dir": "./Dataset",
		"model_path": "./model.ckpt",
		"output_path": "./output.csv",
	}

	return config


def main(
	data_dir,
	model_path,
	output_path,
):
	"""Main function."""
	device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
	print(f"[Info]: Use {device} now!")

	mapping_path = Path(data_dir) / "mapping.json"
	mapping = json.load(mapping_path.open())

	dataset = InferenceDataset(data_dir)
	dataloader = DataLoader(
		dataset,
		batch_size=1,
		shuffle=False,
		drop_last=False,
		num_workers=8,
		collate_fn=inference_collate_fn,
	)
	print(f"[Info]: Finish loading data!",flush = True)

	speaker_num = len(mapping["id2speaker"])
	model = Classifier(n_spks=speaker_num).to(device)
	model.load_state_dict(torch.load(model_path))
	model.eval()
	print(f"[Info]: Finish creating model!",flush = True)

	results = [["Id", "Category"]]
	for feat_paths, mels in tqdm(dataloader):
		with torch.no_grad():
			mels = mels.to(device)
			outs = model(mels)
			preds = outs.argmax(1).cpu().numpy()
			for feat_path, pred in zip(feat_paths, preds):
				results.append([feat_path, mapping["id2speaker"][str(pred)]])

	with open(output_path, 'w', newline='') as csvfile:
		writer = csv.writer(csvfile)
		writer.writerows(results)


if __name__ == "__main__":
	main(**parse_args())
